# Scraper les données de plusieurs pages avec Scrapy

(https://medium.com/@AlexandreWarembourg/scraper-les-donn%C3%A9es-de-plusieurs-pages-avec-scrapy-2e076ac7dc09)


## Démarche de l'article
Le site cible est **MyAnimeList** (`https://myanimelist.net/manga.php`)



## 1. Installation



In [1]:
# Installation de Scrapy 
%pip install scrapy
%pip install nest_asyncio


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Définition du Spider

On reprend la logique de l'article : un spider qui hérite de `scrapy.Spider`, navigue sur les lettres puis pagine sur chaque liste de mangas.

In [2]:
import scrapy
from scrapy import Request


class MangaSpider(scrapy.Spider):
    name = "Manga"
    start_urls = ["https://myanimelist.net/manga.php"]

    # On limite un peu le scraping pour la démo (politesse + rapidité)
    custom_settings = {
        "USER_AGENT": "Mozilla/5.0 (compatible; TutoScrapyBot/1.0)",
        "DOWNLOAD_DELAY": 1.0,         # 1s entre deux requêtes (politesse)
        "ROBOTSTXT_OBEY": False,        # respecte robots.txt
        "CLOSESPIDER_ITEMCOUNT": 100,  # on s'arrête après 100 items pour la démo
        "LOG_LEVEL": "INFO",
    }

    def parse(self, response):
        """Niveau 1 : suit les liens de la navigation alphabétique (A, B, C, ...)."""
        xp = "//div[@id='horiznav_nav']//li/a/@href"
        for url in response.xpath(xp).getall():
            yield Request(response.urljoin(url),
                          callback=self.parse_manga_list_page)

    def parse_manga_list_page(self, response):
        """Niveau 2 : extrait les mangas de la page ET suit la pagination."""
        for row in response.css("div.js-categories-seasonal tr ~ tr"):
            yield {
                "title": self._clean(row.css("a[id] strong::text").get()),
                "synopsis": self._clean(row.css("div.pt4::text").get()),
                "type": self._clean(row.css("td:nth-child(3)::text").get()),
                "volumes": self._clean(row.css("td:nth-child(4)::text").get()),
                "rating": self._clean(row.css("td:nth-child(5)::text").get()),
            }

        # Pagination : on suit le lien vers la page suivante
        for next_url in response.xpath("//div[@class='spaceit']//a/@href").getall():
            yield Request(response.urljoin(next_url),
                          callback=self.parse_manga_list_page)

    @staticmethod
    def _clean(value):
        """Petit utilitaire pour nettoyer les chaînes (évite les None.strip())."""
        return value.strip() if value else None

## 3. Lancement du Spider depuis le notebook

En ligne de commande, l'article fait simplement :
```bash
scrapy crawl Manga -o dataset.jsonlines
```

Dans un notebook, on utilise `CrawlerProcess` pour piloter Scrapy en Python et exporter le résultat en **JSONLines**.


In [3]:
import os
import nest_asyncio
from scrapy.crawler import CrawlerProcess

nest_asyncio.apply()  # autorise Scrapy à utiliser la boucle d'événements de Jupyter

OUTPUT_FILE = "mangas.jsonlines"
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

settings = {
    "FEEDS": {OUTPUT_FILE: {"format": "jsonlines", "encoding": "utf-8"}},
    "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
}

process = CrawlerProcess(settings=settings)
process.crawl(MangaSpider)
process.start()
print("Crawl terminé →", OUTPUT_FILE)

2026-06-01 15:32:57 [scrapy.utils.log] INFO: Scrapy 2.16.0 started (bot: scrapybot)
2026-06-01 15:32:57 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.1',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '26.4.0',
 'Python': '3.13.3 (v3.13.3:6280bb54784, Apr  8 2025, 10:47:54) [Clang 15.0.0 '
           '(clang-1500.3.9.4)]',
 'pyOpenSSL': '26.2.0 (OpenSSL 4.0.0 14 Apr 2026)',
 'cryptography': '48.0.0',
 'Platform': 'macOS-15.5-arm64-arm-64bit-Mach-O'}
2026-06-01 15:32:57 [scrapy.crawler] DEBUG: Using CrawlerProcess
2026-06-01 15:32:57 [scrapy.addons] INFO: Enabled addons:
[]
2026-06-01 15:32:57 [scrapy.extensions.telnet] INFO: Telnet Password: ce31d745e1c8b7f8
2026-06-01 15:32:57 [scrapy.middleware] INFO: Enabled extensions:
['scrapy.extensions.corestats.CoreStats',
 'scrapy.extensions.logcount.LogCount',
 'scrapy.extensions.telnet.TelnetConsole',
 'scrapy.extensions.memusage.MemoryUsage',
 'scrapy.extensions.closespider.CloseSpider',


Crawl terminé → mangas.jsonlines


## 4. Lecture et exploitation des données

Le fichier .jsonlines contient un objet JSON par ligne. On le charge dans un df pour l'explorer.

In [4]:
import json
import pandas as pd

rows = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)
print(f"{len(df)} mangas récupérés.")
df.head(10)

882 mangas récupérés.


,title,synopsis,type,volumes,rating
0,Z,"The ""panic horror"" manga is set in contemporar...",Manga,3,-
1,Z,"The adventures of Agent Z, the newest recruit ...",Manga,2,-
2,Z Boys / Princess,NaN,Light Novel,3,-
3,Z Mazinger,A retelling of Mazinger Z mixed with ancient G...,Manga,5,-
4,Z no Jikan,When hardcore FPS enthusiast Hiroaki Dewa pull...,Light Novel,2,-
5,Z-XL Dai,NaN,One-shot,-,-
6,Z/X: Code Reunion,NaN,Manga,3,-
7,Z/X: Zillions of Enemy X,NaN,Manga,6,-
8,Zaafira Heika to Kuro to Shiro,NaN,Manga,4,-
9,Zaako Zako Zako Zako Sensei,"Souichi Kirisaki, who teaches at a private mid...",Manga,6,6.61


In [5]:
# Quelques statistiques rapides
df.info()
df["type"].value_counts()

<class 'pandas.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   title     882 non-null    str  
 1   synopsis  608 non-null    str  
 2   type      882 non-null    str  
 3   volumes   882 non-null    str  
 4   rating    882 non-null    str  
dtypes: str(5)
memory usage: 34.6 KB


type
Manga          577
Light Novel    115
One-shot        60
Doujinshi       49
Manhwa          47
Manhua          27
Novel            7
Name: count, dtype: int64